# Laboratório — Redes neurais: perceptron, MLP e SOM

**Inteligência Artificial aplicada à Engenharia Elétrica · Projeto 1 · encontros de 11 e 18/11/2026**

Este notebook **complementa** a [página interativa](https://furriel.github.io/templates/inteligencia-artificial/redes-neurais/): lá você manipula as redes; aqui você executa o código C de referência, reproduz os mesmos algoritmos em Python e gera os gráficos que permitem analisar o treinamento (pesos, erros, perdas, gradientes, erro de quantização, ocupação e distâncias).

**Como executar:** baixe o arquivo `.ipynb` e, no Google Colab, use *Arquivo → Fazer upload de notebook*; ou abra-o no Jupyter Notebook/JupyterLab. Não é necessário baixar dados: todos os exemplos são sintéticos e pequenos. As células de C exigem GCC (disponível no Colab); sem GCC elas são puladas com aviso e o restante funciona.

**Coerência com a página:** o Python usa as mesmas equações, sementes e gerador pseudoaleatório dos arquivos `nucleo-*.js` da página e dos programas `perceptron.c` e `mlp_xor.c`. As células de verificação comparam os resultados.

> Material didático. Nenhum dos modelos é sistema de proteção ou diagnóstico de falhas.

## 1. Preparação do ambiente

In [ ]:
import json, math, shutil, subprocess
import numpy as np
import matplotlib.pyplot as plt

GCC = shutil.which("gcc") is not None
print("GCC disponível:", GCC)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})

class LCG:
    # Mesmo gerador da página: s <- (1664525*s + 1013904223) mod 2^32; u = s / 2^32.
    def __init__(self, semente):
        self.s = semente % 2**32
    def proximo(self):
        self.s = (self.s * 1664525 + 1013904223) % 2**32
        return self.s / 4294967296
    def inteiro(self, n):
        return int(self.proximo() * n)

## 2. Perceptron: problema e equações

Dois indicadores binários de um motor — corrente elevada ($x_1$) e temperatura elevada ($x_2$) — devem produzir um alerta didático $d=1$ quando ao menos um estiver ativo (tabela OR). Em um equipamento real, essa regra seria programada diretamente; aqui ela serve para observar **como os coeficientes são aprendidos a partir de exemplos**.

$$z = w_1x_1 + w_2x_2 + b,\qquad y = \begin{cases}1, & z \ge 0\\ 0, & z<0\end{cases},\qquad e = d - y$$

$$w_i \leftarrow w_i + \eta\,e\,x_i,\qquad b \leftarrow b + \eta\,e\qquad(\text{apenas quando } e\neq 0)$$

A regra não deriva o degrau; ela não é retropropagação. Uma **época** percorre os quatro exemplos.

## 3. Cálculo manual e conferência

Com $x=(1,0)$, $w=(-0{,}2;\,0{,}1)$, $b=-0{,}1$, $d=1$, $\eta=0{,}2$: calcule $z$, $y$, $e$ e os novos parâmetros antes de executar a célula.

In [ ]:
x, w, b, d, eta = [1.0, 0.0], [-0.2, 0.1], -0.1, 1, 0.2
z = b
for i in range(2):
    z += w[i] * x[i]
y = int(z >= 0); e = d - y
w_novo = [w[i] + eta * e * x[i] for i in range(2)]
b_novo = b + eta * e
z_novo = b_novo + w_novo[0] * x[0] + w_novo[1] * x[1]
print(f"z = {z:.3f} -> y = {y}; e = {e}; w' = {w_novo}; b' = {b_novo:.3f}; z' = {z_novo:.3f} -> y' = {int(z_novo >= 0)}")
assert abs(z + 0.3) < 1e-12 and y == 0 and e == 1
assert abs(w_novo[0]) < 1e-12 and abs(w_novo[1] - 0.1) < 1e-12 and abs(b_novo - 0.1) < 1e-12
print("CÁLCULO MANUAL CONFERIDO: w2 não mudou porque x2 = 0.")

## 4. O perceptron em C (implementação de referência)

O programa abaixo é o mesmo `perceptron.c` do repositório. A parada ocorre na primeira época sem erros. Os quatro casos testados são os próprios exemplos de treino: **4/4 não demonstra generalização**.

In [ ]:
%%writefile perceptron.c
/*
 * perceptron.c — um neurônio artificial que APRENDE a tabela OR (C11, sem bibliotecas de IA)
 * ------------------------------------------------------------------------------------------
 * Projeto 1 — Inteligência Artificial aplicada à Engenharia Elétrica
 *
 * PROBLEMA: dois indicadores binários de um motor
 *     x1 = corrente acima de um limite didático   (0 ou 1)
 *     x2 = temperatura acima de um limite didático (0 ou 1)
 * devem gerar um alerta d = 1 quando ao menos um for 1 (função lógica OR).
 * Em vez de programar "if (x1 || x2)", deixamos o neurônio descobrir os pesos.
 *
 * EQUAÇÕES
 *     z = b + w1*x1 + w2*x2          soma ponderada (potencial)
 *     y = 1 se z >= 0, senão 0       ativação degrau
 *     erro = d - y                   pode ser -1, 0 ou +1
 *     wi <- wi + eta*erro*xi         regra do perceptron (só muda se errou)
 *     b  <- b  + eta*erro
 *
 * NÃO é um dispositivo de proteção elétrica: é um exercício sobre o mecanismo
 * de aprendizagem. A página da aula reproduz estas contas em JavaScript, na
 * mesma ordem, e um teste automático confere que os dois logs são iguais.
 *
 * Compilar: gcc -std=c11 -Wall -Wextra -pedantic -O2 perceptron.c -o perceptron
 * Executar: ./perceptron        (no Windows: .\perceptron.exe)
 */
#include <stdio.h>

#define N_ENTRADAS 2      /* x1 e x2 */
#define N_AMOSTRAS 4      /* as 4 linhas da tabela OR */
#define MAX_EPOCAS 100    /* limite de segurança: o treino nunca roda para sempre */

/* O "estado" do neurônio: tudo o que ele aprende cabe nestes três números. */
typedef struct {
    double pesos[N_ENTRADAS];   /* w1, w2 */
    double vies;                /* b: desloca a fronteira mesmo quando x = (0,0) */
} Perceptron;

/* z = b + soma(wi * xi). Começa pelo viés e soma produto a produto. */
static double potencial(const Perceptron *p, const double x[N_ENTRADAS]) {
    double z = p->vies;
    for (int i = 0; i < N_ENTRADAS; ++i) {
        z += p->pesos[i] * x[i];
    }
    return z;
}

/* Função degrau: transforma o número real z em uma classe (0 ou 1). */
static int degrau(double z) {
    return z >= 0.0 ? 1 : 0;
}

/* Previsão = degrau(potencial). É o que o neurônio "responde" para x. */
static int prever(const Perceptron *p, const double x[N_ENTRADAS]) {
    return degrau(potencial(p, x));
}

/*
 * Treinamento. Uma ÉPOCA percorre os 4 exemplos em ordem; para cada um:
 *   1. calcula a previsão y;
 *   2. calcula o erro d - y;
 *   3. se errou, corrige pesos e viés imediatamente (o próximo exemplo já
 *      usa os valores corrigidos — por isso a ordem dos exemplos importa).
 * Para quando uma época inteira termina sem erros e devolve o número da época;
 * devolve -1 se atingir max_epocas sem convergir.
 */
static int treinar(Perceptron *p,
                   const double entradas[N_AMOSTRAS][N_ENTRADAS],
                   const int desejado[N_AMOSTRAS],
                   double taxa, int max_epocas) {
    for (int epoca = 1; epoca <= max_epocas; ++epoca) {
        int erros = 0;
        for (int j = 0; j < N_AMOSTRAS; ++j) {
            const int y = prever(p, entradas[j]);
            const int diferenca = desejado[j] - y;          /* erro e = d - y */
            if (diferenca != 0) {
                ++erros;
                for (int i = 0; i < N_ENTRADAS; ++i) {
                    /* se a entrada xi for 0, este produto é 0 e o peso não muda */
                    p->pesos[i] += taxa * diferenca * entradas[j][i];
                }
                p->vies += taxa * diferenca;
            }
        }
        /* Log de uma linha por época: erros encontrados e parâmetros ao FINAL da época. */
        printf("epoca=%02d erros=%d w1=%+.3f w2=%+.3f b=%+.3f\n",
               epoca, erros, p->pesos[0], p->pesos[1], p->vies);
        if (erros == 0) return epoca;                       /* convergiu */
    }
    return -1;                                              /* não convergiu no limite */
}

int main(void) {
    /* Tabela OR: entradas (x1, x2) e saídas desejadas d. */
    const double x[N_AMOSTRAS][N_ENTRADAS] = {
        {0.0, 0.0}, {0.0, 1.0}, {1.0, 0.0}, {1.0, 1.0}
    };
    const int d[N_AMOSTRAS] = {0, 1, 1, 1};
    Perceptron p = {{0.0, 0.0}, 0.0};          /* começa com tudo zero */

    printf("Tabela OR: corrente_alta, temperatura_alta => alerta\n");
    const int epocas = treinar(&p, x, d, 0.2, MAX_EPOCAS);   /* taxa eta = 0,2 */
    if (epocas < 0) {
        fprintf(stderr, "Falha: nao convergiu no limite de epocas.\n");
        return 1;
    }
    printf("Convergiu em %d epocas\n", epocas);

    /* Teste: confere a previsão de cada exemplo. ATENÇÃO: são os mesmos
     * exemplos do treino, então 4/4 mostra que a tabela foi aprendida,
     * não que o modelo generaliza para casos novos. */
    for (int j = 0; j < N_AMOSTRAS; ++j) {
        const double z = potencial(&p, x[j]);
        const int y = prever(&p, x[j]);
        printf("teste=%d x1=%.0f x2=%.0f z=%+.3f previsto=%d esperado=%d\n",
               j, x[j][0], x[j][1], z, y, d[j]);
        if (y != d[j]) return 2;               /* código de saída diferente de 0 = falha */
    }
    printf("TESTES PASSARAM: 4/4\n");
    return 0;
}


In [ ]:
if GCC:
    comp = subprocess.run(["gcc", "-std=c11", "-Wall", "-Wextra", "-pedantic", "-O2", "perceptron.c", "-o", "perceptron"],
                          capture_output=True, text=True)
    print(comp.stdout + comp.stderr)
    assert comp.returncode == 0, "Erro de compilação"
    exec_c = subprocess.run(["./perceptron"], capture_output=True, text=True)
    print(exec_c.stdout)
    assert exec_c.returncode == 0 and "TESTES PASSARAM: 4/4" in exec_c.stdout
else:
    exec_c = None
    print("GCC ausente: execução do C pulada. O restante do notebook continua.")

## 5. O mesmo algoritmo em Python: histórico de pesos, erros e fronteira

A função `treinar_perceptron` repete as operações do C na mesma ordem. Ela registra os parâmetros **a cada passo** e os erros **a cada época**, e detecta ciclos (estado do fim de uma época repetido).

In [ ]:
ENTRADAS = [(0, 0), (0, 1), (1, 0), (1, 1)]
TABELAS = {"OR": [0, 1, 1, 1], "AND": [0, 0, 0, 1], "XOR": [0, 1, 1, 0]}

def treinar_perceptron(tabela, w0=(0.0, 0.0), b0=0.0, eta=0.2, max_epocas=100):
    w, b = list(w0), b0
    passos, epocas, vistos = [(w[0], w[1], b)], [], {}
    for ep in range(1, max_epocas + 1):
        erros = 0
        for x, d in zip(ENTRADAS, TABELAS[tabela]):
            z = b
            for i in range(2):
                z += w[i] * x[i]
            e = d - (1 if z >= 0 else 0)
            if e != 0:
                erros += 1
                for i in range(2):
                    w[i] += eta * e * x[i]
                b += eta * e
            passos.append((w[0], w[1], b))
        epocas.append((ep, erros, w[0], w[1], b))
        if erros == 0:
            return "convergiu", passos, epocas
        chave = tuple(round(v, 9) for v in (w[0], w[1], b))
        if chave in vistos:
            return f"ciclo (repete a época {vistos[chave]})", passos, epocas
        vistos[chave] = ep
    return "limite", passos, epocas

status_or, passos_or, epocas_or = treinar_perceptron("OR")
print("OR:", status_or, "em", len(epocas_or), "épocas")
for ep, er, w1, w2, bb in epocas_or:
    print(f"epoca={ep:02d} erros={er} w1={w1:+.3f} w2={w2:+.3f} b={bb:+.3f}")
if exec_c is not None:
    import re
    log_c = re.findall(r"epoca=(\d+) erros=(\d+) w1=([+-]\d+\.\d+) w2=([+-]\d+\.\d+) b=([+-]\d+\.\d+)", exec_c.stdout)
    log_py = [(f"{ep:02d}", str(er), f"{w1:+.3f}", f"{w2:+.3f}", f"{bb:+.3f}") for ep, er, w1, w2, bb in epocas_or]
    assert log_c == log_py, "Python e C divergiram"
    print("Python reproduz o log do C época a época.")

In [ ]:
passos = np.array(passos_or)
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for k, (nome, cor) in enumerate([("w₁", "#2563a8"), ("w₂", "#0f7b74"), ("b", "#c2570c")]):
    ax[0].step(range(len(passos)), passos[:, k], where="post", label=nome, color=cor)
ax[0].set(title="Parâmetros a cada passo (OR, η = 0,2)", xlabel="passo de treinamento (1 exemplo)", ylabel="valor (adimensional)")
ax[0].legend()
ax[1].bar([e[0] for e in epocas_or], [e[1] for e in epocas_or], color="#b42318")
ax[1].set(title="Erros por época", xlabel="época", ylabel="exemplos classificados errado (de 4)", ylim=(0, 4))
xx, yy = np.meshgrid(np.linspace(-0.3, 1.3, 200), np.linspace(-0.3, 1.3, 200))
for ep, er, w1, w2, bb in epocas_or:
    if abs(w2) > 1e-12:
        ax[2].plot(xx[0], -(w1 * xx[0] + bb) / w2, label=f"fim da época {ep}", alpha=0.4 + 0.6 * ep / len(epocas_or))
    elif abs(w1) > 1e-12:
        ax[2].axvline(-bb / w1, label=f"fim da época {ep}")
for (x1, x2), d in zip(ENTRADAS, TABELAS["OR"]):
    ax[2].scatter(x1, x2, s=160, c="#146e83" if d else "white", edgecolors="#18324b", zorder=3)
    ax[2].annotate(str(d), (x1 + 0.05, x2 + 0.05))
ax[2].set(xlim=(-0.3, 1.3), ylim=(-0.3, 1.3), title="Fronteira w₁x₁ + w₂x₂ + b = 0", xlabel="x₁: corrente elevada", ylabel="x₂: temperatura elevada")
ax[2].legend(fontsize=8)
plt.tight_layout(); plt.show()
w1, w2, bb = epocas_or[-1][2:]
print("Classificação final:", [(x, int(bb + w1 * x[0] + w2 * x[1] >= 0)) for x in ENTRADAS])
print("Observe: (0,1) e (1,0) ficam exatamente sobre a reta (z = 0) e são classe 1 pela convenção z ≥ 0 — solução frágil.")

**Interpretação física.** A reta separa a região “sem alerta” da região “com alerta”. O viés desloca a reta; os pesos a giram. Como dois exemplos ficam exatamente sobre a fronteira, um pequeno ruído numa medição contínua mudaria a decisão — uma margem maior exigiria outro critério de treinamento.

## 6. XOR: a limitação do perceptron simples

XOR exige $(0,1)$ e $(1,0)$ de um lado e $(0,0)$ e $(1,1)$ do outro. Somando as desigualdades: $w_1+w_2+2b \ge 0$ e $w_1+w_2+2b<0$ — contradição. Nenhuma reta separa os dados; o treino entra em ciclo.

In [ ]:
status_xor, passos_xor, epocas_xor = treinar_perceptron("XOR", max_epocas=50)
print("XOR:", status_xor, "após", len(epocas_xor), "épocas")
assert not status_xor.startswith("convergiu")
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].bar([e[0] for e in epocas_xor], [e[1] for e in epocas_xor], color="#b42318")
ax[0].set(title="XOR: erros por época nunca chegam a zero", xlabel="época", ylabel="erros (de 4)", ylim=(0, 4))
p = np.array(passos_xor)
for k, nome in enumerate(["w₁", "w₂", "b"]):
    ax[1].step(range(len(p)), p[:, k], where="post", label=nome)
ax[1].set(title="Parâmetros oscilam em ciclo", xlabel="passo", ylabel="valor")
ax[1].legend(); plt.tight_layout(); plt.show()

## 7. MLP 2–4–1: equações da retropropagação

$$a_j=\sigma\Big(b_{h,j}+\sum_i w_{h,ji}x_i\Big),\quad y=\sigma\Big(b_o+\sum_j w_{o,j}a_j\Big),\quad \sigma(z)=\frac{1}{1+e^{-z}}$$

$$L=-[d\ln y+(1-d)\ln(1-y)],\qquad \delta_o=\frac{\partial L}{\partial z_o}=y-d,\qquad \delta_{h,j}=\delta_o\,w_{o,j}\,a_j(1-a_j)$$

$$\theta \leftarrow \theta-\eta\,\frac{\partial L}{\partial\theta}\quad\text{(treino online, amostra por amostra)}$$

Os deltas são calculados **antes** de alterar os pesos. Acertar os quatro exemplos de treino não prova generalização.

In [ ]:
%%writefile mlp_xor.c
/*
 * mlp_xor.c — rede multicamada 2-4-1 que aprende XOR por retropropagação (C11 + math.h)
 * --------------------------------------------------------------------------------------
 * Projeto 1 — Inteligência Artificial aplicada à Engenharia Elétrica
 *
 * POR QUE ESTA REDE: um perceptron traça uma única reta e não separa XOR
 * ((0,1) e (1,0) de um lado; (0,0) e (1,1) do outro). Com 4 neurônios ocultos,
 * a rede combina várias fronteiras curvas e consegue.
 *
 * ARQUITETURA: 2 entradas -> 4 neurônios ocultos (sigmoide) -> 1 saída (sigmoide)
 *
 * EQUAÇÕES
 *     a_j = sig(bh_j + wh_j0*x0 + wh_j1*x1)       ativação do oculto j
 *     y   = sig(bo + soma_j wo_j*a_j)             saída (interpretada como "classe 1")
 *     sig(z) = 1 / (1 + exp(-z))
 *     L = -[d*ln(y) + (1-d)*ln(1-y)]              perda: entropia cruzada
 *     delta_saida   = y - d                       derivada de L em relação a z da saída
 *     delta_oculta_j = delta_saida * wo_j * a_j * (1 - a_j)   (regra da cadeia)
 *     peso <- peso - taxa * derivada              gradiente descendente
 *
 * ATENÇÃO: todos os deltas são calculados ANTES de atualizar os pesos de
 * saída; caso contrário delta_oculta usaria wo_j já modificado (errado).
 *
 * A página reproduz este programa em JavaScript (mesma semente e ordem) e um
 * teste automático confere: os dois param na época 400 com as mesmas saídas.
 *
 * Compilar: gcc -std=c11 -Wall -Wextra -Werror -pedantic -O2 mlp_xor.c -lm -o mlp_xor
 * (-lm liga a biblioteca matemática, necessária para exp e log)
 */
#include <stdio.h>
#include <math.h>
#include <stdint.h>

#define H 4              /* neurônios ocultos */
#define AMOSTRAS 4       /* linhas da tabela XOR */
#define MAX_EPOCAS 20000 /* limite de segurança */

/* Todos os 17 parâmetros da rede. */
typedef struct {
    double wh[H][2], bh[H], wo[H], bo;
} MLP;

/* Gerador pseudoaleatório (LCG) com semente fixa: a mesma rede inicial a cada
 * execução. Usa os 24 bits mais altos para obter um número em [0, 1). */
static uint32_t estado=11235813u;
static double rand01(void) {
    estado = estado*1664525u+1013904223u;
    return (double)(estado >> 8) / 16777216.0;
}
/* Peso inicial em [-1, 1). */
static double random_weight(void) { return 2.0*rand01()-1.0; }

/* Sigmoide em duas formas equivalentes para evitar estouro de exp() quando z
 * é muito negativo (estabilidade numérica). */
static double sigmoid(double z) {
    if(z>=0.0) return 1.0/(1.0+exp(-z));
    const double ez=exp(z);
    return ez/(1.0+ez);
}
/* Sorteia os pesos sempre na mesma ordem: wh[j][0], wh[j][1], bh[j], wo[j]; por fim bo. */
static void inicializar(MLP *rede) {
    for(int j=0;j<H;j++) {
        for(int i=0;i<2;i++) rede->wh[j][i]=random_weight();
        rede->bh[j]=random_weight();
        rede->wo[j]=random_weight();
    }
    rede->bo=random_weight();
}
/* Propagação direta. Preenche a[] com as ativações ocultas e devolve y. */
static double prever(const MLP *rede,const double x[2],double a[H]) {
    for(int j=0;j<H;j++) {
        double z=rede->bh[j];
        for(int i=0;i<2;i++) z+=rede->wh[j][i]*x[i];
        a[j]=sigmoid(z);
    }
    double z=rede->bo;
    for(int j=0;j<H;j++) z+=rede->wo[j]*a[j];
    return sigmoid(z);
}
/* Perda média (entropia cruzada) nos 4 exemplos. */
static double erro_medio(const MLP *rede,
                         const double x[AMOSTRAS][2],
                         const int d[AMOSTRAS]) {
    double custo=0.0;
    for(int k=0;k<AMOSTRAS;k++) {
        double a[H];double y=prever(rede,x[k],a);
        /* evita log(0) em casos extremos */
        if(y<1e-12) y=1e-12;
        if(y>1.0-1e-12) y=1.0-1e-12;
        custo-=d[k]*log(y)+(1-d[k])*log(1.0-y);
    }
    return custo/AMOSTRAS;
}
/* Um passo de treino com UMA amostra: propaga, retropropaga e atualiza. */
static void passo(MLP *rede,const double x[2],int d,double taxa) {
    double a[H];const double y=prever(rede,x,a);
    /* Sigmoide na saída + entropia cruzada: a derivada simplifica para y - d. */
    const double delta_saida=y-(double)d;
    double delta_oculta[H];
    /* Calcula todos os deltas antes de atualizar pesos de saida. */
    for(int j=0;j<H;j++)
        delta_oculta[j]=delta_saida*rede->wo[j]*a[j]*(1.0-a[j]);
    /* Gradiente descendente na camada de saída... */
    for(int j=0;j<H;j++)rede->wo[j]-=taxa*delta_saida*a[j];
    rede->bo-=taxa*delta_saida;
    /* ...e na camada oculta. */
    for(int j=0;j<H;j++) {
        for(int i=0;i<2;i++)rede->wh[j][i]-=taxa*delta_oculta[j]*x[i];
        rede->bh[j]-=taxa*delta_oculta[j];
    }
}
int main(void) {
    const double x[AMOSTRAS][2]={{0,0},{0,1},{1,0},{1,1}};
    const int d[AMOSTRAS]={0,1,1,0};                    /* tabela XOR */
    MLP rede={0};inicializar(&rede);
    int epoca_parada=-1;
    for(int e=1;e<=MAX_EPOCAS;e++) {
        /* treino "online": atualiza depois de cada amostra, taxa 0,7 */
        for(int k=0;k<AMOSTRAS;k++)passo(&rede,x[k],d[k],0.7);
        /* verifica a perda na 1ª época e depois a cada 100 épocas */
        if(e%100==0 || e==1) {
            const double custo=erro_medio(&rede,x,d);
            if(e%1000==0 || e==1)printf("epoca=%d bce=%.5f\n",e,custo);
            if(custo<0.04) {epoca_parada=e;break;}      /* critério de parada */
        }
    }
    printf("Convergiu (criterio BCE<0.04) na epoca=%d\n",epoca_parada);
    /* Teste nos 4 exemplos (os mesmos do treino: não prova generalização). */
    int corretos=0;
    for(int k=0;k<AMOSTRAS;k++) {
        double a[H],y=prever(&rede,x[k],a);
        const int classe=y>=0.5;                        /* limiar 0,5 */
        printf("x=(%.0f,%.0f) prob=%.4f previsto=%d esperado=%d\n",
               x[k][0],x[k][1],y,classe,d[k]);
        corretos+=(classe==d[k]);
    }
    printf("ACERTOS XOR: %d/4\n",corretos);
    return (epoca_parada>0 && corretos==AMOSTRAS)?0:1;  /* 0 = sucesso */
}


In [ ]:
if GCC:
    comp = subprocess.run(["gcc", "-std=c11", "-Wall", "-Wextra", "-Werror", "-pedantic", "-O2", "mlp_xor.c", "-lm", "-o", "mlp_xor"],
                          capture_output=True, text=True)
    print(comp.stdout + comp.stderr)
    assert comp.returncode == 0
    exec_mlp = subprocess.run(["./mlp_xor"], capture_output=True, text=True)
    print(exec_mlp.stdout)
    assert exec_mlp.returncode == 0 and "ACERTOS XOR: 4/4" in exec_mlp.stdout
else:
    exec_mlp = None
    print("GCC ausente: execução do C pulada.")

## 8. A mesma MLP em Python: perda, ativações, gradientes e superfície de decisão

A implementação usa o mesmo gerador (`rand01` do C), a mesma ordem de inicialização e a mesma função exponencial da biblioteca matemática; por isso a época de parada e as saídas coincidem com o C. Uma verificação por diferenças finitas confirma os gradientes.

In [ ]:
XOR = [((0, 0), 0), ((0, 1), 1), ((1, 0), 1), ((1, 1), 0)]
H = 4

def criar_mlp(semente=11235813):
    s = semente
    def r01():
        nonlocal s
        s = (s * 1664525 + 1013904223) % 2**32
        return (s >> 8) / 16777216.0
    rede = {"wh": [], "bh": [], "wo": [], "bo": 0.0}
    for j in range(H):
        rede["wh"].append([2 * r01() - 1, 2 * r01() - 1])
        rede["bh"].append(2 * r01() - 1)
        rede["wo"].append(2 * r01() - 1)
    rede["bo"] = 2 * r01() - 1
    return rede

def sigmoide(z):
    if z >= 0:
        return 1.0 / (1.0 + math.exp(-z))
    ez = math.exp(z)
    return ez / (1.0 + ez)

def forward(rede, x):
    a = []
    for j in range(H):
        z = rede["bh"][j]
        for i in range(2):
            z += rede["wh"][j][i] * x[i]
        a.append(sigmoide(z))
    z = rede["bo"]
    for j in range(H):
        z += rede["wo"][j] * a[j]
    return a, sigmoide(z)

def perda(y, d):
    y = min(max(y, 1e-12), 1 - 1e-12)
    return -(d * math.log(y) + (1 - d) * math.log(1 - y))

def gradientes(rede, x, d):
    a, y = forward(rede, x)
    do = y - d
    dh = [do * rede["wo"][j] * a[j] * (1 - a[j]) for j in range(H)]   # pesos ANTES da atualização
    return {"wo": [do * a[j] for j in range(H)], "bo": do,
            "wh": [[dh[j] * x[0], dh[j] * x[1]] for j in range(H)], "bh": dh}, a, y

def passo(rede, x, d, eta):
    g, a, y = gradientes(rede, x, d)
    for j in range(H):
        rede["wo"][j] -= eta * g["wo"][j]
    rede["bo"] -= eta * g["bo"]
    for j in range(H):
        for i in range(2):
            rede["wh"][j][i] -= eta * g["wh"][j][i]
        rede["bh"][j] -= eta * g["bh"][j]
    return g

def perda_media(rede):
    return sum(perda(forward(rede, x)[1], d) for x, d in XOR) / 4

def vetor(rede):
    return [v for j in range(H) for v in rede["wh"][j]] + rede["bh"] + rede["wo"] + [rede["bo"]]

def de_vetor(v):
    return {"wh": [[v[2 * j], v[2 * j + 1]] for j in range(H)], "bh": v[8:12], "wo": v[12:16], "bo": v[16]}

# verificação numérica do gradiente (diferenças centrais)
r0 = criar_mlp()
pior = 0.0
for x, d in XOR:
    g = gradientes(r0, x, d)[0]
    gv = [v for j in range(H) for v in g["wh"][j]] + g["bh"] + g["wo"] + [g["bo"]]
    v = vetor(r0)
    for k in range(17):
        mais, menos = v.copy(), v.copy(); mais[k] += 1e-6; menos[k] -= 1e-6
        num = (perda(forward(de_vetor(mais), x)[1], d) - perda(forward(de_vetor(menos), x)[1], d)) / 2e-6
        pior = max(pior, abs(num - gv[k]))
print(f"Maior diferença entre gradiente analítico e numérico: {pior:.2e}")
assert pior < 1e-6

In [ ]:
rede = criar_mlp(); rede_inicial = json.loads(json.dumps(rede))
hist_perda, hist_grad, epoca_parada = [perda_media(rede)], [], None
for ep in range(1, 20001):
    norma = 0.0
    for x, d in XOR:
        g = passo(rede, x, d, 0.7)
        norma += sum(abs(v) for v in g["wo"]) + abs(g["bo"]) + sum(abs(v) for l in g["wh"] for v in l) + sum(abs(v) for v in g["bh"])
    hist_perda.append(perda_media(rede)); hist_grad.append(norma)
    if (ep % 100 == 0 or ep == 1) and hist_perda[-1] < 0.04:
        epoca_parada = ep
        break
print("Critério BCE < 0,04 atingido na época", epoca_parada)
probs = [forward(rede, x)[1] for x, _ in XOR]
for (x, d), p in zip(XOR, probs):
    print(f"x={x} y={p:.4f} classe={int(p >= 0.5)} esperado={d}")
if exec_mlp is not None:
    import re
    assert int(re.search(r"epoca=(\d+)\n", exec_mlp.stdout).group(1)) == epoca_parada
    assert re.findall(r"prob=(\d\.\d{4})", exec_mlp.stdout) == [f"{p:.4f}" for p in probs]
    print("Python reproduz o C: mesma época de parada e mesmas saídas (4 casas).")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(hist_perda, color="#b42318"); ax[0].axhline(0.04, ls=":", color="k", label="critério 0,04")
ax[0].set(title="Perda média por época", xlabel="época", ylabel="entropia cruzada média (nats)"); ax[0].legend()
ax[1].semilogy(range(1, len(hist_grad) + 1), hist_grad, color="#7c3aed")
ax[1].set(title="Soma de |gradiente| por época", xlabel="época", ylabel="Σ|∂L/∂θ| (escala log)")
A = np.array([forward(rede, x)[0] for x, _ in XOR])
im = ax[2].imshow(A, cmap="Blues", vmin=0, vmax=1)
ax[2].set(title="Ativações ocultas após o treino", xticks=range(4), xticklabels=["a₁", "a₂", "a₃", "a₄"],
          yticks=range(4), yticklabels=[f"x={x}, d={d}" for x, d in XOR])
for i in range(4):
    for j in range(4):
        ax[2].text(j, i, f"{A[i, j]:.2f}", ha="center", va="center", color="white" if A[i, j] > 0.6 else "black")
fig.colorbar(im, ax=ax[2], shrink=0.8, label="aⱼ")
plt.tight_layout(); plt.show()

g = np.linspace(-0.25, 1.25, 120)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.6))
for k, (r, titulo) in enumerate([(rede_inicial, "Antes do treino"), (rede, f"Após {epoca_parada} épocas")]):
    Z = np.array([[forward(r, (a, b))[1] for a in g] for b in g])
    c = ax[k].contourf(g, g, Z, levels=np.linspace(0, 1, 11), cmap="BrBG")
    ax[k].contour(g, g, Z, levels=[0.5], colors="k", linewidths=2)
    for x, d in XOR:
        ax[k].scatter(*x, s=150, c="#146e83" if d else "white", edgecolors="k", zorder=3)
    ax[k].set(title=titulo + " — saída y(x₁, x₂)", xlabel="x₁", ylabel="x₂")
fig.colorbar(c, ax=ax, shrink=0.85, label="y (estimativa da classe 1)")
plt.show()
print("A linha preta (y = 0,5) deixa de ser uma reta: a camada oculta combina várias fronteiras.")

## 9. SOM em duas dimensões: ver a grade se desdobrar

Antes dos caracteres, um exemplo em que os pesos podem ser desenhados no próprio plano das entradas. Cada vértice da malha é o vetor de pesos de um neurônio; as linhas ligam vizinhos **na grade**. Os pontos vêm de dois regimes fictícios; as cores não entram no treino.

$$c=\arg\min_j\|x-m_j\|^2,\qquad h(c,j)=e^{-\|r_c-r_j\|^2/(2\sigma^2)},\qquad m_j\leftarrow m_j+\alpha\,h(c,j)\,(x-m_j)$$

In [ ]:
rng = np.random.default_rng(20261104)
dados2 = np.vstack([np.clip(rng.normal([.26, .31], .095, (60, 2)), .04, .96), np.clip(rng.normal([.73, .70], .095, (60, 2)), .04, .96)])
origem = np.array([0] * 60 + [1] * 60)
grade = 6
pos2 = np.array([(r, c) for r in range(grade) for c in range(grade)])
pesos2_0 = np.random.default_rng(8472).uniform(.12, .88, (grade * grade, 2)); pesos2 = pesos2_0.copy()
eq2 = []
for k in range(2000):
    x = dados2[k % len(dados2)]
    c = int(np.argmin(np.sum((pesos2 - x) ** 2, axis=1)))
    t = min(k / 1700, 1)
    alfa, sig = .28 * (.05 ** t), 2.5 * (.24 ** t)
    pesos2 += alfa * np.exp(-np.sum((pos2 - pos2[c]) ** 2, axis=1) / (2 * sig ** 2))[:, None] * (x - pesos2)
    if k % 50 == 0:
        eq2.append(np.mean(np.sqrt(np.min(((dados2[:, None] - pesos2[None]) ** 2).sum(2), axis=1))))
fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))
for a, P, titulo in ((ax[0], pesos2_0, "Antes do treino"), (ax[1], pesos2, "Após 2000 amostras")):
    a.scatter(dados2[:, 0], dados2[:, 1], c=origem, s=12, alpha=.45, cmap="coolwarm")
    for j, (r, c) in enumerate(pos2):
        if c + 1 < grade: a.plot(*P[[j, j + 1]].T, "k-", lw=1, alpha=.6)
        if r + 1 < grade: a.plot(*P[[j, j + grade]].T, "k-", lw=1, alpha=.6)
    a.set(xlim=(0, 1), ylim=(0, 1), title=titulo, xlabel="corrente normalizada (adim.)", ylabel="velocidade normalizada (adim.)"); a.set_aspect("equal")
ax[2].plot(np.arange(0, 2000, 50), eq2, color="#146e83")
ax[2].set(title="Erro de quantização", xlabel="atualizações", ylabel="distância média à BMU (adim.)")
plt.tight_layout(); plt.show()

## 10. SOM de caracteres 5×7 — a mesma implementação do simulador

Cada dígito é uma matriz de 5 colunas × 7 linhas, lida por linhas como um vetor de 35 valores binários. A SOM tem 25 neurônios (grade 5×5), cada um com 35 pesos.

- **Treino (não supervisionado):** glifo sorteado e, com probabilidade 0,75, um pixel invertido. O nome do glifo não entra na atualização.
- **Esquemas:** $\alpha(t)=0{,}30\,e^{-t/800}+0{,}02$; $\sigma(t)=1{,}6\,e^{-t/600}+0{,}35$.
- **Critério de parada:** a cada 100 atualizações, $|\Delta EQ|\le 0{,}004$ e variação média dos pesos $\le 0{,}004$ por 3 verificações seguidas; limite de segurança de 20 000. Não usa rótulos nem o dígito escolhido.
- **Calibração (depois do treino):** 360 padrões rotulados (glifo limpo + 35 versões com um pixel invertido) votam em suas BMUs. Não altera pesos.
- **Teste:** 500 padrões com dois pixels invertidos, que não aparecem no treino nem na calibração.

As funções abaixo replicam `nucleo-som.js` (mesmo gerador, mesma ordem). A célula de verificação grava `som_notebook_resultado.json`, usado pelos testes automáticos para comparar com o navegador.

In [ ]:
FONTE = {
 "0": ["01110","10001","10011","10101","11001","10001","01110"], "1": ["00100","01100","00100","00100","00100","00100","01110"],
 "2": ["01110","10001","00001","00010","00100","01000","11111"], "3": ["11110","00001","00001","01110","00001","00001","11110"],
 "4": ["00010","00110","01010","10010","11111","00010","00010"], "5": ["11111","10000","10000","11110","00001","00001","11110"],
 "6": ["01110","10000","10000","11110","10001","10001","01110"], "7": ["11111","00001","00010","00100","01000","01000","01000"],
 "8": ["01110","10001","10001","01110","10001","10001","01110"], "9": ["01110","10001","10001","01111","00001","00001","01110"]}
DIGITOS = list(FONTE)
PADROES = {d: np.array([int(c) for c in "".join(FONTE[d])], dtype=float) for d in DIGITOS}
POS = np.array([(j // 5, j % 5) for j in range(25)], dtype=float)
CFG = dict(semente=20261104, alfa0=.30, alfaMin=.02, tauAlfa=800, sigma0=1.6, sigmaMin=.35, tauSigma=600,
           probRuido=.75, intervalo=100, tolEQ=.004, tolPesos=.004, paciencia=3, maxAtualizacoes=20000)

def inverter(x, i):
    y = x.copy(); y[i] = 1 - y[i]; return y
CAL_X = np.array([PADROES[d] if i < 0 else inverter(PADROES[d], i) for d in DIGITOS for i in range(-1, 35)])
CAL_D = [d for d in DIGITOS for _ in range(36)]

def alfa(t, c=CFG): return c["alfa0"] * math.exp(-t / c["tauAlfa"]) + c["alfaMin"]
def sigma(t, c=CFG): return c["sigma0"] * math.exp(-t / c["tauSigma"]) + c["sigmaMin"]

def metricas(W, X=CAL_X):
    D = ((X[:, None, :] - W[None]) ** 2).sum(2)
    o = np.argsort(D, axis=1, kind="stable")
    adj = np.abs(POS[o[:, 0]] - POS[o[:, 1]]).sum(1) == 1
    return float(np.mean(np.sqrt(D[np.arange(len(X)), o[:, 0]]))), float(np.mean(~adj))

def treinar_som(cfg=CFG, registrar=()):
    rng = LCG(cfg["semente"])
    W = np.array([[0.08 + 0.84 * rng.proximo() for _ in range(35)] for _ in range(25)])
    t, estaveis, ant = 0, 0, W.copy()
    eq0, et0 = metricas(W); eq_ant = eq0
    verif, snaps = [(0, eq0, et0, None, None)], {0: W.copy()}
    status = "limite"
    while t < cfg["maxAtualizacoes"]:
        d = DIGITOS[rng.inteiro(10)]
        x = PADROES[d].copy()
        if rng.proximo() < cfg["probRuido"]:
            x = inverter(x, rng.inteiro(35))
        a, s = alfa(t, cfg), sigma(t, cfg)
        c = int(np.argmin(((x - W) ** 2).sum(1)))
        h = np.exp(-((POS - POS[c]) ** 2).sum(1) / (2 * s * s))
        W += a * h[:, None] * (x - W)
        t += 1
        if t in registrar: snaps[t] = W.copy()
        if t % cfg["intervalo"] == 0:
            eq, et = metricas(W)
            deq, dw = abs(eq - eq_ant), float(np.mean(np.abs(W - ant)))
            estavel = deq <= cfg["tolEQ"] and dw <= cfg["tolPesos"]
            estaveis = estaveis + 1 if estavel else 0
            eq_ant, ant = eq, W.copy()
            verif.append((t, eq, et, deq, dw))
            if estaveis >= cfg["paciencia"]:
                status = "convergiu"; break
    snaps[t] = W.copy()
    return W, t, status, verif, snaps

def calibrar(W):
    votos = np.zeros((25, 10), dtype=int)
    for x, d in zip(CAL_X, CAL_D):
        votos[int(np.argmin(((x - W) ** 2).sum(1))), DIGITOS.index(d)] += 1
    rot = [DIGITOS[int(np.argmax(v))] if v.max() > 0 else None for v in votos]
    return rot, votos

## 11. Treinar até o critério: evolução do erro e dos pesos

In [ ]:
W, t_final, status, verif, snaps = treinar_som(registrar={1, 10, 100, 300, 1000})
print(f"Status: {status} em {t_final} atualizações (limite {CFG['maxAtualizacoes']}).")
assert status == "convergiu", "Critério não atingido: analise a falha antes de interpretar o mapa."
json.dump({"atualizacoes": t_final, "pesos": W.tolist()}, open("som_notebook_resultado.json", "w"))

v = np.array([(a, b, c, np.nan if d is None else d, np.nan if e is None else e) for a, b, c, d, e in verif], dtype=float)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(v[:, 0], v[:, 1], label="erro de quantização EQ", color="#146e83")
ax[0].plot(v[:, 0], v[:, 2], ls="--", label="erro topográfico (fração)", color="#b45309")
ax[0].set(title="Qualidade do mapa", xlabel="atualizações", ylabel="EQ (distância em 35 dim.) · ET (fração)"); ax[0].legend()
ax[1].semilogy(v[1:, 0], v[1:, 3], label="|ΔEQ| entre verificações", color="#146e83")
ax[1].semilogy(v[1:, 0], v[1:, 4], label="variação média dos pesos", color="#7c3aed")
ax[1].axhline(CFG["tolEQ"], ls=":", color="#b42318", label="tolerância")
ax[1].set(title="Critério de parada", xlabel="atualizações", ylabel="variação (escala log)"); ax[1].legend()
plt.tight_layout(); plt.show()

def desenhar_mapa(ax, W, titulo, rotulos=None):
    img = np.ones((5 * 8, 5 * 6))
    for j in range(25):
        r, c = divmod(j, 5)
        img[r * 8:r * 8 + 7, c * 6:c * 6 + 5] = 1 - W[j].reshape(7, 5)
    ax.imshow(img, cmap="gray", vmin=0, vmax=1)
    if rotulos:
        for j, rot in enumerate(rotulos):
            r, c = divmod(j, 5)
            ax.text(c * 6 + 2, r * 8 + 7.6, rot if rot else "—", ha="center", fontsize=8, color="#b45309")
    ax.set(title=titulo, xticks=[], yticks=[])

fig, ax = plt.subplots(1, len(snaps), figsize=(3.2 * len(snaps), 3.6))
for a, (t, Wt) in zip(ax, sorted(snaps.items())):
    desenhar_mapa(a, Wt, f"t = {t}")
fig.suptitle("Os 25 vetores de pesos (escuro = peso próximo de 1): do aleatório aos padrões")
plt.tight_layout(); plt.show()

## 12. Calibração, ocupação das unidades e mapa de distâncias

- **Rótulos:** associados depois do treino, por votação; unidades sem votos ficam sem rótulo.
- **Ocupação:** quantos padrões de calibração escolhem cada unidade como BMU.
- **Matriz U:** distância média entre os pesos de cada unidade e os de suas vizinhas na grade; valores altos marcam fronteiras entre grupos.

In [ ]:
rotulos, votos = calibrar(W)
ocup = votos.sum(1).reshape(5, 5)
U = np.zeros((5, 5))
for j in range(25):
    viz = [k for k in range(25) if np.abs(POS[k] - POS[j]).sum() == 1]
    U[j // 5, j % 5] = np.mean([np.linalg.norm(W[j] - W[k]) for k in viz])
fig, ax = plt.subplots(1, 3, figsize=(16, 4.8))
desenhar_mapa(ax[0], W, f"Pesos finais e rótulos (t = {t_final})", rotulos)
im = ax[1].imshow(ocup, cmap="Blues"); fig.colorbar(im, ax=ax[1], shrink=.8, label="nº de padrões de calibração")
for j in range(25): ax[1].text(j % 5, j // 5, f"{ocup[j // 5, j % 5]}\nrót. {rotulos[j] or '—'}", ha="center", va="center", fontsize=9, color="white" if ocup[j // 5, j % 5] > 18 else "black")
ax[1].set(title="Ocupação das unidades (BMU)", xlabel="coluna da grade", ylabel="linha da grade")
im = ax[2].imshow(U, cmap="magma_r"); fig.colorbar(im, ax=ax[2], shrink=.8, label="distância média às vizinhas (35 dim.)")
ax[2].set(title="Matriz U (fronteiras entre grupos)", xlabel="coluna da grade", ylabel="linha da grade")
plt.tight_layout(); plt.show()
print("Unidades rotuladas:", sum(r is not None for r in rotulos), "de 25")

## 13. Reconhecimento dos dez dígitos, ruído e falhas

O reconhecimento compara a entrada com **todos** os 25 neurônios, escolhe a menor distância (BMU) e só então consulta o rótulo. Os pesos não mudam.

In [ ]:
def reconhecer(W, x):
    d2 = ((x - W) ** 2).sum(1)
    o = np.argsort(d2, kind="stable")
    return int(o[0]), rotulos[int(o[0])], float(np.sqrt(d2[o[0]])), float(np.sqrt(d2[o[1]]) - np.sqrt(d2[o[0]])), d2

W_antes = W.copy()
print("Glifos limpos:")
for d in DIGITOS:
    c, rot, dist, margem, _ = reconhecer(W, PADROES[d])
    print(f"  {d} -> BMU N{c + 1:02d}, rótulo {rot}, d = {dist:.3f}, margem = {margem:.3f}")
    assert rot == d

rng_t = LCG(7)                      # mesmo conjunto de teste da página
teste = []
for d in DIGITOS:
    for _ in range(50):
        i = rng_t.inteiro(35); j = rng_t.inteiro(34)
        if j >= i: j += 1
        teste.append((inverter(inverter(PADROES[d], i), j), d, (i, j)))
saidas = [reconhecer(W, x)[1] for x, _, _ in teste]
acertos = sum(s == d for s, (_, d, _) in zip(saidas, teste))
print(f"\nTeste com 2 pixels invertidos: {acertos}/{len(teste)} = {100 * acertos / len(teste):.1f} % (base sintética, não OCR real)")
assert np.array_equal(W, W_antes), "o reconhecimento não pode alterar pesos"

conf = np.zeros((10, 11), dtype=int)
for s, (_, d, _) in zip(saidas, teste):
    conf[DIGITOS.index(d), 10 if s is None else DIGITOS.index(s)] += 1
falhas = [(x, d, px, s) for s, (x, d, px) in zip(saidas, teste) if s != d]
fig, ax = plt.subplots(1, 2, figsize=(14, 4.8), gridspec_kw={"width_ratios": [1.2, 1]})
im = ax[0].imshow(conf, cmap="Blues")
ax[0].set(title="Matriz de confusão (teste com 2 pixels invertidos)", xlabel="saída (— = sem rótulo)", ylabel="glifo de origem",
          xticks=range(11), xticklabels=DIGITOS + ["—"], yticks=range(10), yticklabels=DIGITOS)
for i in range(10):
    for j in range(11):
        if conf[i, j]: ax[0].text(j, i, conf[i, j], ha="center", va="center", color="white" if conf[i, j] > 25 else "black")
ax[1].axis("off"); ax[1].set_title(f"Primeiras falhas ({len(falhas)} no total)")
for k, (x, d, px, s) in enumerate(falhas[:8]):
    a = fig.add_axes([0.58 + (k % 4) * 0.095, 0.45 - (k // 4) * 0.38, 0.08, 0.3])
    a.imshow(1 - x.reshape(7, 5), cmap="gray", vmin=0, vmax=1); a.set(xticks=[], yticks=[])
    a.set_title(f"{d}→{s or '—'}", fontsize=9)
plt.show()

**Como ler as falhas.** Uma falha não é “erro do código”: duas inversões podem aproximar um glifo de outro (por exemplo, 8 e 9 diferem em poucos pixels), ou levar a BMU a uma unidade sem rótulo. A distância não é probabilidade e a rede não rejeita entradas desconhecidas: uma matriz vazia também tem BMU.

## 14. Das redes deste laboratório aos modelos de linguagem

| | Perceptron | MLP | SOM | Transformer / LLM |
|---|---|---|---|---|
| Parâmetros | 3 | 17 | 875 | bilhões |
| Treino | regra do perceptron | gradiente (retropropagação) | competição + vizinhança, sem rótulos | gradiente para prever o próximo token |
| Inferência | um limiar | composição de não linearidades | vetor de referência mais próximo | atenção entre posições e geração token a token |

Todos ajustam parâmetros no treino e os mantêm fixos na inferência. A BMU é uma escolha por distância a referências fixas; a atenção é uma soma ponderada aprendida entre posições do contexto. Uma SOM não modela sequências nem evolui para um Transformer.

## 15. Exercícios e entrega

1. Refaça o cálculo manual com $x=(0,1)$, $w=(0{,}3;\,-0{,}4)$, $b=0{,}1$, $d=1$, $\eta=0{,}1$ e confira com a página (modo Inferência + Treinamento).
2. Treine o perceptron em OR com $\eta \in \{0{,}1; 0{,}5; 1{,}0\}$ partindo de zeros. Compare épocas e a posição final da reta. Por que η maior não garante menos épocas?
3. Explique, com as desigualdades da seção 6, por que XOR entra em ciclo.
4. Na MLP, altere a semente em `criar_mlp` (por exemplo 1, 2, 3) e relate quais convergem até 20 000 épocas. O que significa “limite” no simulador?
5. Na SOM, use `treinar_som({**CFG, "tolEQ": 0, "tolPesos": 0, "maxAtualizacoes": 3000})`. O que o status informa? Por que esse resultado não pode ser apresentado como convergente?
6. Escolha uma falha da seção 13, desenhe a entrada e a BMU, e explique a troca usando as distâncias.

Respostas esperadas e rubrica: [Exercícios e rubricas](https://furriel.github.io/templates/inteligencia-artificial/estudante/exercicios.html).